# 9.12 · 分布式训练 / Distributed Training

> **课程定位 / Where this fits**
> 第 12 课，**Part 9 · 深度学习基础**。
> Lesson 12, **Part 9 · Deep Learning Foundations**.
>
> 当模型/数据大到单张 GPU 装不下或训练太慢，就要用**多 GPU、多机器并行**。这是大模型时代的核心工程。本节偏概念（多卡环境无法在本地真跑），讲清**数据并行 / 模型并行 / 流水线并行**的区别、DDP 的工作原理(梯度 all-reduce)、以及常见名词，并用单机模拟梯度平均建立直觉。
> When a model/data is too big for one GPU or too slow, use **multi-GPU, multi-machine parallelism** — core engineering of the large-model era. This lesson is concept-leaning (multi-GPU can't truly run locally): the difference between **data / model / pipeline parallel**, how DDP works (gradient all-reduce), common terms, plus a single-process simulation of gradient averaging.
>
> 💼 **实战/面试视角**："数据并行 vs 模型并行 / DDP 怎么同步梯度 / all-reduce 是什么" 是大模型/基础设施岗高频。
> 💼 **Practical/interview angle:** "data vs model parallel / how DDP syncs gradients / what's all-reduce" — frequent for large-model/infra roles.

> 📐 **符号约定 / Notation**
> - GPU/worker —— 一个计算设备/进程 / one compute device or process
> - all-reduce —— 把各 worker 的梯度求和/平均再广播回去 / sum/average grads across workers and broadcast back

> 💡 **面试相关 / Interview-relevant**
> - "数据并行 vs 模型并行的区别"（出镜率 ★★★★★）
> - "DDP 如何保持各卡模型一致"（★★★★，梯度 all-reduce）
> - "为什么 DDP 比 DataParallel 好"（★★★）
> - "大模型放不下单卡怎么办"（★★★★，模型/流水线/张量并行 + ZeRO）

---

## 学习目标 / Learning Objectives
1. 区分**数据并行 / 模型并行 / 流水线并行**及各自适用场景。
   Distinguish data / model / pipeline parallel and when to use each.
2. 理解 **DDP** 用梯度 **all-reduce** 保持各卡一致。
   Understand DDP keeps GPUs in sync via gradient all-reduce.
3. 知道 **DataParallel vs DistributedDataParallel** 的差别。
   Know DataParallel vs DistributedDataParallel.
4. 了解大模型扩展技术（ZeRO/张量并行）的思路。
   Know the ideas behind large-model scaling (ZeRO/tensor parallel).

## 目录 / TOC
1. [为什么要分布式 ⭐](#1)
2. [三种并行方式 ⭐](#2)
3. [数据并行与 all-reduce（模拟）⭐](#3)
4. [DDP 的写法 + 大模型扩展 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么要分布式 ⭐ / Why Distributed

两个瓶颈逼我们用多设备：
Two bottlenecks force multi-device:
- **太慢**：数据/模型不算太大，但单卡训一遍要好几天。多卡**各算一部分数据**能近线性加速。
  **Too slow:** data/model fit, but one GPU takes days per run. Multiple GPUs each handle **part of the data** for near-linear speedup.
- **太大**：模型本身(参数+激活+优化器状态)装不进单卡显存。必须把**模型切开**放到多卡。
  **Too big:** the model itself (params + activations + optimizer state) won't fit one GPU's memory. The **model must be split** across GPUs.

这对应两大思路：**数据并行**(解决"太慢")和**模型并行**(解决"太大")。现代大模型训练常常**两者结合**。
These map to two ideas: **data parallel** (for "too slow") and **model parallel** (for "too big"). Modern large-model training often **combines both**.


<a id="2"></a>
## 2. 三种并行方式 ⭐ / Three Kinds of Parallelism

**① 数据并行(Data Parallelism)**——最常用。**每张卡都有一份完整模型副本**，但各自处理**不同的数据子集**。各卡独立前向+反向算出梯度，然后**把梯度平均(all-reduce)同步**，保证所有副本始终一致。适合"模型装得下、但想训得快"。
**① Data Parallelism** — most common. **Each GPU holds a full model copy** but processes a **different data subset**. Each computes gradients independently, then **averages gradients (all-reduce) to sync**, keeping all copies identical. For "model fits, want speed."

**② 模型并行(Model Parallelism)**——**把模型本身切成几段，分别放到不同卡**（比如前 20 层在 GPU0、后 20 层在 GPU1）。一个 batch 的数据依次流过各卡。适合"模型大到单卡装不下"。缺点：同一时刻只有一张卡在算，其它在等。
**② Model Parallelism** — **split the model itself across GPUs** (e.g. first 20 layers on GPU0, next 20 on GPU1). A batch flows through GPUs in sequence. For "model too big for one GPU." Downside: only one GPU computes at a time, others idle.

**③ 流水线并行(Pipeline Parallelism)**——模型并行的改进：把 batch 再切成**微批(micro-batch)**像流水线一样喂进去，让各卡**同时**处理不同微批，减少空等。
**③ Pipeline Parallelism** — improved model parallel: split the batch into **micro-batches** fed like an assembly line so GPUs work **simultaneously** on different micro-batches, reducing idle time.

> **张量并行(Tensor Parallelism)** 是更细的模型并行：把**单个大矩阵乘法**也拆到多卡。大模型(如 LLM)常把数据+流水线+张量三种并行叠加用("3D 并行")。
> **Tensor Parallelism** is finer model parallel: split **a single big matmul** across GPUs. Large models (LLMs) often stack data + pipeline + tensor ("3D parallelism").


<a id="3"></a>
## 3. 数据并行与 all-reduce（模拟）⭐ / Data Parallel & All-Reduce (Simulated)

数据并行的核心是 **all-reduce 梯度同步**。我们在**单进程里模拟 N 张卡**：把一个大 batch 分给 N 个"worker"，每个 worker 在自己的数据上算梯度，然后把所有 worker 的梯度**平均**——这平均后的梯度，等于直接在整个大 batch 上算的梯度。这正是数据并行**数学上等价于大 batch 单卡训练**的原因。
The heart of data parallel is **all-reduce gradient sync**. We **simulate N GPUs in one process**: split a big batch among N "workers," each computes gradients on its shard, then **average** all workers' gradients — this average equals the gradient computed on the whole big batch. That's why data parallel is **mathematically equivalent to large-batch single-GPU training**.


In [ ]:
import numpy as np
import torch, torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data/16.0, digits.target, test_size=0.3,
                                          stratify=digits.target, random_state=0)
ce = nn.CrossEntropyLoss()

# --- 方式A: 整个大 batch 在"单卡"上算梯度 / single GPU on the whole big batch ---
torch.manual_seed(0); net_single = nn.Linear(64, 10)
xb = torch.tensor(X_tr[:128], dtype=torch.float32); yb = torch.tensor(y_tr[:128])
ce(net_single(xb), yb).backward()
grad_single = net_single.weight.grad.clone()              # 单卡梯度 / single-GPU gradient

# --- 方式B: 把 128 个样本分给 4 个 worker, 各算梯度再平均(all-reduce) / 4 workers, average grads ---
torch.manual_seed(0); net_dp = nn.Linear(64, 10)
n_workers = 4
shards = torch.chunk(xb, n_workers); yshards = torch.chunk(yb, n_workers)  # 切成4份 / split into 4 shards
worker_grads = []
for xs, ys in zip(shards, yshards):                       # 每个 worker 独立算梯度 / each worker computes grads
    net_dp.zero_grad(); ce(net_dp(xs), ys).backward()
    worker_grads.append(net_dp.weight.grad.clone())
grad_avg = torch.stack(worker_grads).mean(0)              # all-reduce: 平均各 worker 梯度 / average across workers

diff = (grad_single - grad_avg).abs().max().item()
print(f"单卡(整批)梯度 与 4-worker平均梯度 的最大差异 = {diff:.2e}")
print("→ 几乎为 0: 数据并行(各卡算梯度后平均)在数学上等价于大 batch 单卡训练")
print("这就是 all-reduce: 把各 worker 的梯度求和/平均, 同步回所有 worker, 保持模型副本一致")


<a id="4"></a>
## 4. DDP 的写法 + 大模型扩展 + 小结 ⭐ / DDP Code, Scaling & Summary

PyTorch 里数据并行有两种 API：
PyTorch offers two data-parallel APIs:
- **`nn.DataParallel`（旧，不推荐）**：单进程多线程，主卡负担重、有 Python GIL 瓶颈、效率低。
  **`nn.DataParallel` (old, discouraged):** single-process multi-thread, heavy on the main GPU, GIL-bound, inefficient.
- **`nn.parallel.DistributedDataParallel` (DDP，推荐)**：**每张卡一个独立进程**，梯度在反向传播时就用 NCCL 高效 all-reduce，几乎线性加速。**面试问"为什么 DDP 比 DataParallel 好"答这个**。
  **`DistributedDataParallel` (DDP, recommended):** **one process per GPU**, gradients all-reduced efficiently via NCCL during backward, near-linear scaling. **This answers "why DDP > DataParallel."**

下面是 DDP 的标准骨架（仅展示写法，需在多 GPU + `torchrun` 下真跑）：
Below is the DDP skeleton (pattern only; needs multi-GPU + `torchrun` to actually run):


In [ ]:
# DDP 标准骨架(伪代码/写法展示, 单机CPU不真跑) / DDP skeleton (pattern only)
ddp_template = """
import torch, os
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler

def main():
    dist.init_process_group(backend="nccl")          # 初始化进程组(GPU 用 nccl) / init process group
    rank = dist.get_rank()                            # 当前进程编号 / this process id
    local = rank % torch.cuda.device_count()
    torch.cuda.set_device(local)                      # 每个进程绑一张卡 / bind one GPU per process

    model = MyModel().to(local)
    model = DDP(model, device_ids=[local])            # 包一层 DDP: 反向时自动 all-reduce 梯度 / auto all-reduce

    sampler = DistributedSampler(dataset)             # 保证各进程拿到不重叠的数据子集 / non-overlapping shards
    loader = DataLoader(dataset, sampler=sampler, batch_size=64)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(EPOCHS):
        sampler.set_epoch(epoch)                      # 每轮重新打乱 / reshuffle each epoch
        for xb, yb in loader:
            xb, yb = xb.to(local), yb.to(local)
            opt.zero_grad(); loss = loss_fn(model(xb), yb)
            loss.backward()                           # DDP 在此自动同步梯度(all-reduce) / grads synced here
            opt.step()
    dist.destroy_process_group()

# 启动: torchrun --nproc_per_node=4 train.py  (4 卡, 每卡一个进程)
"""
print("DDP 关键点:")
print("1) 每 GPU 一个进程(不是多线程), 用 torchrun 启动")
print("2) DDP 包裹模型, backward 时自动 all-reduce 同步梯度 → 各卡模型始终一致")
print("3) DistributedSampler 保证各进程数据不重叠")
print("4) 比旧的 nn.DataParallel 快很多(无 GIL/主卡瓶颈)")


**模型太大装不下单卡**时（如大语言模型），除了模型/流水线/张量并行，还有一类省显存技术：
**When a model is too big for one GPU** (e.g. LLMs), beyond model/pipeline/tensor parallel, there are memory-saving techniques:
- **ZeRO / FSDP**：把**优化器状态、梯度、甚至参数**分片到各卡，每张卡只存一部分，需要时再聚合。能在同样的卡上训练大得多的模型。
  **ZeRO / FSDP:** shard **optimizer states, gradients, even parameters** across GPUs; each holds only a slice, gathered when needed. Trains far larger models on the same hardware.
- **梯度检查点(gradient checkpointing)**：前向时不存所有中间激活，反向时重算，**用计算换显存**（与 9.11 的省显存技巧互补）。
  **Gradient checkpointing:** don't store all activations in forward, recompute in backward, **trading compute for memory** (complements 9.11's memory tricks).

```
为什么分布式: 太慢→数据并行(各卡算部分数据); 太大→模型并行(切开模型)
数据并行: 每卡完整模型副本+不同数据, 梯度 all-reduce 平均同步 → 等价大batch
模型并行: 把模型切段放不同卡(模型装不下时); 流水线并行用micro-batch减少空等; 张量并行切单个矩阵乘
DDP(推荐) vs DataParallel(旧): 每卡一进程 vs 单进程多线程; DDP无GIL/主卡瓶颈, 近线性加速
大模型扩展: ZeRO/FSDP分片优化器状态/梯度/参数; 梯度检查点用计算换显存; LLM常3D并行(数据+流水线+张量)
```

### 💡 面试速查 / Interview cheat-sheet
1. **数据并行**: 复制模型, 分数据, all-reduce 平均梯度(解决慢)。
   Data parallel: replicate model, split data, all-reduce gradients (for speed).
2. **模型/流水线/张量并行**: 切模型(解决装不下); 流水线用 micro-batch 减空等。
   Model/pipeline/tensor parallel: split model (for size); pipeline uses micro-batches.
3. **all-reduce**: 各卡梯度求和/平均再广播, 保持副本一致。
   All-reduce: sum/average grads across GPUs and broadcast, keeping copies in sync.
4. **DDP > DataParallel**: 每卡一进程, 无 GIL/主卡瓶颈, 近线性加速。
   DDP > DataParallel: one process per GPU, no GIL/main-GPU bottleneck, near-linear.
5. **大模型省显存**: ZeRO/FSDP 分片 + 梯度检查点。
   Large-model memory: ZeRO/FSDP sharding + gradient checkpointing.

### 下一节 / Next
**9.13 迁移学习**——从零训练很贵。用在大数据集上预训练好的模型, 在自己的小数据上微调, 是实战中最实用、最高性价比的技巧。
**9.13 Transfer Learning** — training from scratch is costly. Take a model pretrained on big data, fine-tune on your small data — the most practical, cost-effective trick in practice.
